# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/professormyhre/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/professormyhre/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Use-Case Data!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [5]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

/Users/professormyhre/aie8/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
/Users/professormyhre/aie8/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/lang/arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
/Users/professormyhre/aie8/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/lang/persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [7]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [8]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 64, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [9]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

Property 'summary' already exists in node 'a08b8e'. Skipping!
Property 'summary' already exists in node 'aea656'. Skipping!
Property 'summary' already exists in node '0402e3'. Skipping!
Property 'summary' already exists in node '078267'. Skipping!
Property 'summary' already exists in node '751359'. Skipping!
Property 'summary' already exists in node 'b16721'. Skipping!
Property 'summary' already exists in node 'b106c7'. Skipping!
Property 'summary' already exists in node '7f0308'. Skipping!
Property 'summary' already exists in node 'f12966'. Skipping!
Property 'summary' already exists in node 'aebe65'. Skipping!
Property 'summary' already exists in node 'e2278e'. Skipping!
Property 'summary' already exists in node 'c5d8c5'. Skipping!
Property 'summary' already exists in node '049577'. Skipping!
Property 'summary' already exists in node '4f1a9f'. Skipping!
Property 'summary' already exists in node 'd6c8ca'. Skipping!
Property 'summary' already exists in node '104e83'. Skipping!
Property

Applying CustomNodeFilter:   0%|          | 0/8 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/48 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node '0402e3'. Skipping!
Property 'summary_embedding' already exists in node '751359'. Skipping!
Property 'summary_embedding' already exists in node 'a08b8e'. Skipping!
Property 'summary_embedding' already exists in node '078267'. Skipping!
Property 'summary_embedding' already exists in node 'b16721'. Skipping!
Property 'summary_embedding' already exists in node 'aea656'. Skipping!
Property 'summary_embedding' already exists in node 'aebe65'. Skipping!
Property 'summary_embedding' already exists in node '049577'. Skipping!
Property 'summary_embedding' already exists in node '4423f0'. Skipping!
Property 'summary_embedding' already exists in node 'e2278e'. Skipping!
Property 'summary_embedding' already exists in node '7f0308'. Skipping!
Property 'summary_embedding' already exists in node 'c5d8c5'. Skipping!
Property 'summary_embedding' already exists in node 'f12966'. Skipping!
Property 'summary_embedding' already exists in node '4f1a9f'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 86, relationships: 712)

We can save and load our knowledge graphs as follows.

In [10]:
kg.save("usecase_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase_data_kg.json")
usecase_data_kg

KnowledgeGraph(nodes: 86, relationships: 712)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [11]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [12]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

# 1. SingleHopSpecificQuerySynthesizer: This synthesizer creates straightforward, fact-based questions that can be answered using a single piece of information from the knowledge graph. Think of it as generating simple, direct questions like "What is X?" or "Who did Y?"

# 2. MultiHopAbstractQuerySynthesizer: This one generates more complex questions that require connecting multiple pieces of information (multiple "hops" in the knowledge graph) and often asks about broader or more abstract concepts. For example, it might create questions like "How does A relate to B in the context of C?"

# 3. MultiHopSpecificQuerySynthesizer: This synthesizer also creates questions that require combining information from several parts of the knowledge graph, but the questions are more specific and detailed rather than abstract. For instance, it might ask, "What steps did X take to achieve Y, and what was the result?"

# In summary:  
# - *SingleHopSpecific* = simple, direct questions  
# - *MultiHopAbstract* = complex, broad/abstract questions  
# - *MultiHopSpecific* = complex, detailed/specific questions



Finally, we can use our `TestSetGenerator` to generate our testset!

In [14]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,"According to Bick et al., 2024, what are the k...",[Introduction ChatGPT launched in November 202...,"Bick et al., 2024, build on previous work by a...",single_hop_specifc_query_synthesizer
1,How does ChatGPT usage differ between work-rel...,[Table 1: ChatGPT daily message counts (millio...,Table 1 indicates that while total daily messa...,single_hop_specifc_query_synthesizer
2,SOC what is it,[Variation by Occupation Figure 23 presents va...,"The context does not explicitly define SOC, bu...",single_hop_specifc_query_synthesizer
3,Who is Brynjolfsson in relation to AI research?,[Conclusion This paper studies the rapid growt...,The context mentions Collis and Brynjolfsson (...,single_hop_specifc_query_synthesizer
4,Hoo is the percentege distrbution of mesages i...,[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,"In June 2024, the percentage of non-work messa...",multi_hop_abstract_query_synthesizer
5,how work vs non-work use and message asking do...,[<1-hop>\n\nConclusion This paper studies the ...,the paper says most chatgpt msgs r non-work (7...,multi_hop_abstract_query_synthesizer
6,how fast ChatGPT grow and how generative AI te...,[<1-hop>\n\nConclusion This paper studies the ...,The paper studies the rapid growth of ChatGPT ...,multi_hop_abstract_query_synthesizer
7,how many messages 18 billion and 2.5 billion m...,[<1-hop>\n\nConclusion This paper studies the ...,The first context segment states that by July ...,multi_hop_specific_query_synthesizer
8,How does the growth of ChatGPT usage in the US...,[<1-hop>\n\nTable 1: ChatGPT daily message cou...,"The context indicates that in the US, ChatGPT'...",multi_hop_specific_query_synthesizer
9,"Based on the findings of Handa et al. (2025), ...",[<1-hop>\n\nTable 1: ChatGPT daily message cou...,Handa et al. (2025) report that while nearly 8...,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [15]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

Property 'summary' already exists in node '6f668a'. Skipping!
Property 'summary' already exists in node 'd4fef9'. Skipping!
Property 'summary' already exists in node '9ddd90'. Skipping!
Property 'summary' already exists in node 'a0e37b'. Skipping!
Property 'summary' already exists in node '9c3125'. Skipping!
Property 'summary' already exists in node '4b3a00'. Skipping!
Property 'summary' already exists in node '035ebe'. Skipping!
Property 'summary' already exists in node '9d4e0d'. Skipping!
Property 'summary' already exists in node '30c865'. Skipping!
Property 'summary' already exists in node 'ecc124'. Skipping!
Property 'summary' already exists in node '2dd60d'. Skipping!
Property 'summary' already exists in node 'f73ce3'. Skipping!
Property 'summary' already exists in node 'e8b83c'. Skipping!
Property 'summary' already exists in node 'daf47a'. Skipping!
Property 'summary' already exists in node '63e0af'. Skipping!
Property 'summary' already exists in node '7a845e'. Skipping!
Property

...

Property 'summary_embedding' already exists in node '9ddd90'. Skipping!
Property 'summary_embedding' already exists in node 'a0e37b'. Skipping!
Property 'summary_embedding' already exists in node '9c3125'. Skipping!
Property 'summary_embedding' already exists in node 'd4fef9'. Skipping!
Property 'summary_embedding' already exists in node '4b3a00'. Skipping!
Property 'summary_embedding' already exists in node '6f668a'. Skipping!
Property 'summary_embedding' already exists in node '9d4e0d'. Skipping!
Property 'summary_embedding' already exists in node '30c865'. Skipping!
Property 'summary_embedding' already exists in node 'e8b83c'. Skipping!
Property 'summary_embedding' already exists in node 'f73ce3'. Skipping!
Property 'summary_embedding' already exists in node '035ebe'. Skipping!
Property 'summary_embedding' already exists in node 'ecc124'. Skipping!
Property 'summary_embedding' already exists in node '63e0af'. Skipping!
Property 'summary_embedding' already exists in node '2dd60d'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [16]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,"Bick et al., 2024 what they say about ChatGPT ...",[Introduction ChatGPT launched in November 202...,"The paper by Bick et al., 2024, studies consum...",single_hop_specifc_query_synthesizer
1,How does OpenAI contribute to the development ...,[Table 1: ChatGPT daily message counts (millio...,OpenAI is involved in understanding product us...,single_hop_specifc_query_synthesizer
2,What information is available about Appendix D...,[Variation by Occupation Figure 23 presents va...,Appendix D contains a full report of GWA count...,single_hop_specifc_query_synthesizer
3,Who is Brynjolfsson in relation to this research?,[Conclusion This paper studies the rapid growt...,The context references Collis and Brynjolfsson...,single_hop_specifc_query_synthesizer
4,How does AI impact productivity outside work a...,[<1-hop>\n\nTable 1: ChatGPT daily message cou...,The context indicates that while most AI analy...,multi_hop_abstract_query_synthesizer
5,H0w has the rapid growht and adoption of ChatG...,[<1-hop>\n\nConclusion This paper studies the ...,"The rapid growth and adoption of ChatGPT, laun...",multi_hop_abstract_query_synthesizer
6,Based on the data about ChatGPT's rapid growth...,[<1-hop>\n\nConclusion This paper studies the ...,The data indicates that ChatGPT's usage has be...,multi_hop_abstract_query_synthesizer
7,Based on the variation in ChatGPT usage by occ...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,The data indicates that users in highly paid p...,multi_hop_abstract_query_synthesizer
8,"Based on the data from the US, how has ChatGPT...",[<1-hop>\n\nConclusion This paper studies the ...,"The data indicates that in the US, ChatGPT's u...",multi_hop_specific_query_synthesizer
9,WHy is US usage of ChatGPT so high and how doe...,[<1-hop>\n\nConclusion This paper studies the ...,The context indicates that ChatGPT's usage in ...,multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [17]:
from langsmith import Client

client = Client()

dataset_name = "Use Case Synthetic Data - AIE8"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [18]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [19]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [20]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [21]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [22]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG"
)

In [23]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [24]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

As is usual: We'll be using `gpt-4.1-mini` for our RAG!

In [25]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [26]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [27]:
rag_chain.invoke({"question" : "What are people doing with AI these days?"})

'Based on the provided context, people are using AI, particularly generative AI like ChatGPT, in a variety of ways. They perform workplace tasks by augmenting or automating human labor. AI is used to produce writing, software code, spreadsheets, and other digital products, which differentiates it from traditional search engines. Users engage with AI for different intents such as asking questions, doing tasks, or expressing themselves. Additionally, AI serves both as a co-worker producing output and as a co-pilot giving advice and improving human productivity. There are also uses related to self-expression (like relationships, personal reflection, games, and role play), though these represent a smaller portion of AI use. Overall, AI is highly flexible and applied across professional, creative, and personal domains.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [28]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [29]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

dopeness_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "dopeness": "Is this response dope, lit, cool, or is it just a generic response?",
        },
        "llm" : eval_llm
    }
)

# #### 🏗️ Activity #2:
#
# Highlight what each evaluator is evaluating.
#
# - `qa_evaluator`: Checks if the answer is correct.
# - `labeled_helpfulness_evaluator`: Checks if the answer is helpful.
# - `dopeness_evaluator`: Checks if the answer is unique and cool.

## LangSmith Evaluation

In [30]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'excellent-egg-72' at:
https://smith.langchain.com/o/64d1906c-49dc-4ba3-ad6b-466372cc69b9/datasets/e603ecbe-2a0f-4858-a617-ba315b70257a/compare?selectedSessions=18e72249-9a71-4750-ba5d-e25f06ed2cfc




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,What does the data from July 2025 reveal about...,The data from July 2025 reveals that since Cha...,None,The data from July 2025 shows that ChatGPT has...,1,0,0,3.334186,5edd162c-83be-4f1e-b94a-5b93256e21f6,eadf3efa-e291-4cc0-80d5-fa9f34b70bf2
1,how november 2022 chatgpt launch and the growt...,"ChatGPT launched publicly on November 30, 2022...",None,The context shows that ChatGPT launched in Nov...,1,1,0,6.682690,fa69fbcf-6b8f-4d55-b042-4e11f126fc91,00a96009-02b1-465c-9d6c-6a4ddd6e78c3
2,WHy is US usage of ChatGPT so high and how doe...,The context does not provide specific informat...,None,The context indicates that ChatGPT's usage in ...,0,0,0,3.990335,75985b37-6d6a-4bbb-b065-8ba9c8b4c8ba,285de5c6-163b-46b2-baf4-34bcfe811b4c
3,"Based on the data from the US, how has ChatGPT...","Based on the provided context, ChatGPT experie...",None,"The data indicates that in the US, ChatGPT's u...",1,1,0,3.788598,784cdf8b-a80d-4960-a9ef-f86aad9af010,314bceac-16ca-4099-9cee-41c1c77f3afb
4,Based on the variation in ChatGPT usage by occ...,"Based on the provided context, usage patterns ...",None,The data indicates that users in highly paid p...,1,1,0,5.273020,720768d1-a618-4267-aee7-f89ac2e42885,bd06812b-13c4-4c1f-b380-0eb838216614
5,Based on the data about ChatGPT's rapid growth...,Based on the provided context:\n\n- **Gender**...,None,The data indicates that ChatGPT's usage has be...,1,1,0,5.895010,82d552c6-cfdf-4b16-be9a-a0f1405837e4,a4af527c-3618-4335-b444-8c6166b138cd
6,H0w has the rapid growht and adoption of ChatG...,"The rapid growth and adoption of ChatGPT, laun...",None,"The rapid growth and adoption of ChatGPT, laun...",1,1,0,4.335579,acb02af9-31bc-4741-ae65-5e4e0b3bdf27,55677042-e87c-401f-bab6-10c6e2af6ade
7,How does AI impact productivity outside work a...,"Based on the provided context:\n\nAI, includin...",None,The context indicates that while most AI analy...,1,1,0,4.093555,480d885b-4a89-4cf9-86fc-e6cbfcd8a4e8,0dca1c7e-9553-407d-8793-61106fc81644
8,Who is Brynjolfsson in relation to this research?,"Based on the provided context, Brynjolfsson (E...",None,The context references Collis and Brynjolfsson...,1,0,0,2.251170,e39a6f2a-7d77-4a33-ab76-bca5f0580a37,24433923-a230-4b7a-92ea-37b6e9128ab9
9,What information is available about Appendix D...,Appendix D contains a full report of Generaliz...,None,Appendix D contains a full report of GWA count...,1,1,0,2.174379,ae370c56-0d07-427d-899a-69f5f0e26691,040d40c6-3428-41f5-bee3-ce7ee225b274


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [31]:
DOPENESS_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Make your answer rad, ensure high levels of dopeness. Do not be generic, or give generic responses.

Context: {context}
Question: {question}
"""

dopeness_rag_prompt = ChatPromptTemplate.from_template(DOPENESS_RAG_PROMPT)

In [32]:
rag_documents = docs

In [33]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

# 
# Modifying the chunk size changes how much context is included in each document chunk that the retriever and LLM see. 
# 
# - If the chunk size is too small, important information may be split across multiple chunks, making it harder for the model to retrieve all relevant context for a question. This can lead to incomplete or less accurate answers.
# - If the chunk size is too large, each chunk may contain a lot of unrelated or noisy information, which can confuse the retriever or the LLM, and may also exceed model input limits.
# 
# Therefore, tuning the chunk size helps balance between providing enough context for accurate answers and avoiding irrelevant or excessive information, directly impacting retrieval and answer quality.


In [34]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

# 
# Modifying the embedding model changes how your application represents and compares text data. Different embedding models capture different aspects of language, such as semantic meaning, context, or domain-specific knowledge. A more powerful or better-aligned embedding model can improve retrieval accuracy by making similar questions and documents closer in vector space, leading to more relevant results. Conversely, a less suitable embedding model may cause relevant information to be missed or irrelevant information to be retrieved, reducing answer quality. Therefore, the choice of embedding model directly impacts how well your application understands and matches user queries to the right context.


In [35]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG Docs"
)

In [36]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [37]:
dopeness_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dopeness_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [38]:
dopeness_rag_chain.invoke({"question" : "How are people using AI to make money?"})

'Alright, listen up—the AI money-making game is straight-up next-level. From the context, folks aren’t just letting AI do tasks on autopilot; they’re jamming it as a boss-level advisor or research sidekick that supercharges decision-making. Especially in knowledge-crunching jobs, ChatGPT isn’t just punching the clock, it’s dropping decision support heat, helping workers make smarter moves that level up their productivity and, by extension, their cash flow. \n\nAnd here’s the kicker: Collis and Brynjolfsson’s deep-dive tells us users in the US would literally pay up tens of billions yearly *to keep* this AI juice flowing. That means AI isn’t just a tool—it’s a massive value generator, pumping at least $97 billion in surplus just from making work smarter, not harder. So the money magic? It’s coming from AI being this slick advisor that boosts income by amping up how well people work, think, and decide. Straight cash flow through brainpower amplification. Pure future-tech hustle. Boom.'

Finally, we can evaluate the new chain on the same test set!

In [39]:
evaluate(
    dopeness_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "dopeness_rag_chain"},
)

View the evaluation results for experiment: 'notable-vessel-49' at:
https://smith.langchain.com/o/64d1906c-49dc-4ba3-ad6b-466372cc69b9/datasets/e603ecbe-2a0f-4858-a617-ba315b70257a/compare?selectedSessions=b587790b-88bb-4b11-9a86-a1c12804b9e5




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,What does the data from July 2025 reveal about...,"Yo, buckle up because the ChatGPT launch story...",None,The data from July 2025 shows that ChatGPT has...,1,0,1,6.533346,5edd162c-83be-4f1e-b94a-5b93256e21f6,e77bde7d-ee40-4228-a7f4-9e47d404f820
1,how november 2022 chatgpt launch and the growt...,"Alright, strap in for a turbo-charged blast th...",None,The context shows that ChatGPT launched in Nov...,1,1,1,8.027479,fa69fbcf-6b8f-4d55-b042-4e11f126fc91,9ee6d779-c7f9-4234-b302-200e6320b456
2,WHy is US usage of ChatGPT so high and how doe...,"Yo, let's break down the epic vibes behind US ...",None,The context indicates that ChatGPT's usage in ...,1,1,1,3.739159,75985b37-6d6a-4bbb-b065-8ba9c8b4c8ba,c6bb274e-7eed-4972-8a5a-364bc94a280c
3,"Based on the data from the US, how has ChatGPT...","Yo, let’s unpack this with some serious swag s...",None,"The data indicates that in the US, ChatGPT's u...",1,1,1,5.788041,784cdf8b-a80d-4960-a9ef-f86aad9af010,f8a78e5a-fc83-40a8-934d-80d6101d2cc5
4,Based on the variation in ChatGPT usage by occ...,"Alright, buckle up for this deep dive into the...",None,The data indicates that users in highly paid p...,1,1,1,7.830788,720768d1-a618-4267-aee7-f89ac2e42885,7d6c81f6-2286-4a70-adb6-96216d4fb508
5,Based on the data about ChatGPT's rapid growth...,"Yo, here’s the lowdown straight from the ChatG...",None,The data indicates that ChatGPT's usage has be...,1,1,1,7.001966,82d552c6-cfdf-4b16-be9a-a0f1405837e4,a0518e00-28ae-4112-afc9-c3b414c70ff4
6,H0w has the rapid growht and adoption of ChatG...,"Yo, here’s the turbo-charged scoop straight fr...",None,"The rapid growth and adoption of ChatGPT, laun...",1,1,1,9.625811,acb02af9-31bc-4741-ae65-5e4e0b3bdf27,8a87a536-c3d4-4565-b285-120ec87667e2
7,How does AI impact productivity outside work a...,"Alright, strap in for this AI power play! 🚀\n\...",None,The context indicates that while most AI analy...,1,1,1,5.043715,480d885b-4a89-4cf9-86fc-e6cbfcd8a4e8,a67134e2-07f0-476f-b4f7-8f36706faf2f
8,Who is Brynjolfsson in relation to this research?,"Yo, here’s the deal on Brynjolfsson straight f...",None,The context references Collis and Brynjolfsson...,1,1,1,1.831881,e39a6f2a-7d77-4a33-ab76-bca5f0580a37,d85bedff-ed6d-4438-8f68-cf6bb694c94d
9,What information is available about Appendix D...,"Oh, get ready for the nitty-gritty straight fr...",None,Appendix D contains a full report of GWA count...,1,1,1,4.091810,ae370c56-0d07-427d-899a-69f5f0e26691,21516d8e-0b91-4630-9bf2-335e9e64553d


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

